# LendInsight — Notebook 3: Risk Segmentation
**Purpose**: Validate and analyze the rule-based risk segmentation (LOW / MEDIUM / HIGH).  
**Segmentation Logic**: Based on loan grade + DTI proxy (loan_percent_income).  
**Output**: `credit_risk_segmented.csv` — used as Power BI data source.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.family": "sans-serif"})

CLEAN_PATH = r"C:\data_analyst\LendInsight\01_data\clean\credit_risk_cleaned.csv"
df = pd.read_csv(CLEAN_PATH)
print(f"Dataset: {df.shape[0]:,} rows | risk_category already derived by ETL pipeline.")
print(df["risk_category"].value_counts())


## 1. Risk Segmentation Logic
The `risk_category` column was created in the ETL pipeline using this business rule:

| Condition | Category |
|-----------|----------|
| Grade E/F/G **or** DTI > 0.40 | **HIGH** |
| Grade A/B **and** DTI < 0.20 | **LOW** |
| Everything else | **MEDIUM** |

This mirrors how a credit risk team would classify borrowers — using the internal risk grade as the primary signal and DTI as a secondary stress indicator.


In [ ]:

# Confirm distribution
seg = df["risk_category"].value_counts()
total = len(df)
print("Risk Category Distribution:")
print("-" * 35)
for cat, cnt in seg.items():
    print(f"  {cat:<8}: {cnt:>6,}  ({cnt/total*100:.1f}%)")


## 2. Validation — Default Rate per Segment
This confirms the segmentation is meaningful: HIGH risk must default more than LOW risk.

In [ ]:

seg_df = df.groupby("risk_category").agg(
    total=("default_flag","count"),
    defaults=("default_flag","sum"),
    avg_loan=("loan_amnt","mean"),
    avg_rate=("loan_int_rate","mean"),
    avg_dti=("loan_percent_income","mean")
).reset_index()
seg_df["default_rate"] = (seg_df["defaults"] / seg_df["total"] * 100).round(2)
seg_df["pct_portfolio"] = (seg_df["total"] / total * 100).round(1)

display_df = seg_df[["risk_category","total","pct_portfolio","defaults","default_rate","avg_loan","avg_rate","avg_dti"]].copy()
display_df.columns = ["Segment","Loans","% Portfolio","Defaults","Default Rate %","Avg Loan","Avg Rate %","Avg DTI"]
print(display_df.to_string(index=False))


## 3. Visualization — Risk Segment Performance

In [ ]:

order = ["LOW","MEDIUM","HIGH"]
seg_ordered = seg_df.set_index("risk_category").reindex(order).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = {"LOW":"#2ECC71","MEDIUM":"#F39C12","HIGH":"#E74C3C"}
color_list = [colors[c] for c in order]

# Default rate
axes[0].bar(order, seg_ordered["default_rate"], color=color_list, edgecolor="white", width=0.5)
axes[0].set_title("Default Rate by Risk Segment", fontweight="bold")
axes[0].set_ylabel("Default Rate (%)")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, (cat, val) in enumerate(zip(order, seg_ordered["default_rate"])):
    axes[0].text(i, val+0.5, f"{val:.1f}%", ha="center", fontweight="bold")

# Loan count (donut)
axes[1].pie(seg_ordered["total"], labels=order, colors=color_list,
            autopct="%1.1f%%", startangle=90,
            wedgeprops=dict(edgecolor="white", linewidth=2, width=0.6))
axes[1].set_title("Portfolio Share by Risk Segment", fontweight="bold")

# Avg loan amount
axes[2].bar(order, seg_ordered["avg_loan"], color=color_list, edgecolor="white", width=0.5)
axes[2].set_title("Avg Loan Amount by Risk Segment", fontweight="bold")
axes[2].set_ylabel("Avg Loan (USD)")
for i, val in enumerate(seg_ordered["avg_loan"]):
    axes[2].text(i, val+50, f"${val:,.0f}", ha="center", fontweight="bold", fontsize=9)

plt.suptitle("Risk Segmentation Validation", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_13_risk_segments.png", bbox_inches="tight")
plt.show()


## 4. Risk Category vs Loan Intent Heatmap
**Question**: Where does HIGH risk concentrate by loan purpose?

In [ ]:

pivot = df.groupby(["risk_category","loan_intent"])["default_flag"].mean().unstack() * 100
pivot = pivot.reindex(["HIGH","MEDIUM","LOW"])

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Default Rate (%)"})
ax.set_title("Default Rate Heatmap: Risk Category × Loan Intent",
             fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("Risk Category"); ax.set_xlabel("Loan Intent")
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_14_segment_intent_heatmap.png", bbox_inches="tight")
plt.show()


## 5. TOP 10 Highest-Risk Customer Profiles

In [ ]:

profile_df = df.groupby(["loan_grade","loan_intent","income_bracket","dti_bracket"]).agg(
    count=("default_flag","count"),
    defaults=("default_flag","sum")
).reset_index()
profile_df["default_rate"] = (profile_df["defaults"]/profile_df["count"]*100).round(1)
profile_df = profile_df[profile_df["count"] >= 20].sort_values("default_rate", ascending=False).head(10)

print("TOP 10 Riskiest Customer Profiles (min 20 loans per group):")
print("-"*80)
print(profile_df[["loan_grade","loan_intent","income_bracket","dti_bracket","count","defaults","default_rate"]].to_string(index=False))


## 6. Export Segmented Dataset for Power BI

In [ ]:

OUTPUT_PATH = r"C:\data_analyst\LendInsight\01_data\clean\credit_risk_segmented.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Segmented dataset exported:")
print(f"  Path : {OUTPUT_PATH}")
print(f"  Rows : {len(df):,}")
print(f"  Cols : {len(df.columns)}")
print(f"\nColumns: {list(df.columns)}")
print("\nThis file is the Power BI data source.")


## 7. Key Findings Summary

| Finding | Detail |
|---------|--------|
| **Overall default rate** | 21.87% — critically high |
| **Grade G default rate** | Significantly above portfolio average |
| **DTI > 0.40 borrowers** | Default at the highest rate in any bracket |
| **Prior defaulters** | Default again at a much higher rate than clean borrowers |
| **HIGH risk segment** | Represents a small share of volume but concentrated losses |
| **Income gap** | Defaulted borrowers have meaningfully lower average income |
| **Loan intent** | Certain purposes (e.g. VENTURE) carry above-average default risk |

> **Interview talking point**: *"The risk segmentation shows that grade and DTI together are strong predictors of default — which justifies the business rule approach over a complex model for this use case."*
